# Anomaly Detection Evaluation

This notebook evaluates the anomaly detection system on synthetic data with labeled anomalies.


In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.metrics import precision_score, recall_score, f1_score

from src.ap.ingest.simulator import MarketEvent
from src.ap.processing.features import FeatureStore
from src.ap.models.isolation_forest import IsolationForestDetector


In [ ]:
# Generate synthetic data with labeled anomalies
def generate_labeled_data(n_normal=1000, n_anomalies=100):
    """Generate synthetic data with known anomalies."""
    events = []
    labels = []
    
    base_time = datetime.now()
    base_price = 100.0
    
    # Normal events
    for i in range(n_normal):
        price = base_price + np.random.normal(0, 0.5)
        size = np.random.uniform(0.5, 5.0)
        events.append({
            'timestamp': base_time + timedelta(seconds=i),
            'symbol': 'TEST',
            'price': price,
            'size': size,
            'side': np.random.choice(['buy', 'sell'])
        })
        labels.append(0)
    
    # Anomaly events (volume spikes)
    for i in range(n_anomalies):
        price = base_price + np.random.normal(0, 0.5)
        size = np.random.uniform(20, 50)  # Large volume
        events.append({
            'timestamp': base_time + timedelta(seconds=n_normal + i),
            'symbol': 'TEST',
            'price': price,
            'size': size,
            'side': np.random.choice(['buy', 'sell'])
        })
        labels.append(1)
    
    return pd.DataFrame(events), np.array(labels)


In [ ]:
# Generate data
df, true_labels = generate_labeled_data()
print(f"Generated {len(df)} events, {true_labels.sum()} anomalies")


In [ ]:
# Process events and compute features
feature_store = FeatureStore()
features_list = []
event_labels = []

for idx, row in df.iterrows():
    event = MarketEvent(
        timestamp=row['timestamp'],
        symbol=row['symbol'],
        price=row['price'],
        size=row['size'],
        side=row['side']
    )
    feature_store.update(event)
    features = feature_store.compute_features(event.symbol)
    
    if features:
        features_list.append(features)
        event_labels.append(true_labels[idx])


In [ ]:
# Split train/test
split_idx = len(features_list) // 2
train_features = features_list[:split_idx]
test_features = features_list[split_idx:]
test_labels = event_labels[split_idx:]

print(f"Train: {len(train_features)}, Test: {len(test_features)}")


In [ ]:
# Train Isolation Forest
detector = IsolationForestDetector()
detector.fit(train_features)

# Evaluate
predictions = []
scores = []

for features in test_features:
    score = detector.score(features)
    is_anomaly = detector.is_anomaly(features)
    predictions.append(1 if is_anomaly else 0)
    scores.append(score)

# Metrics
precision = precision_score(test_labels, predictions)
recall = recall_score(test_labels, predictions)
f1 = f1_score(test_labels, predictions)

print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
